LET'S FIRST START BY INSTALLING ALL REQUIRED PACAKGES FOR THIS PROJECT

In [ ]:
!pip install -q \
transformers\
datasets\
accelerate\
bitsandbytes\
peft\
trl\
unsloth\
mlflow\
tqdm\
numpy\
scipy\
sentencepiece\
safetensors\
tensorboard\
wandb\
rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#let's define our model name and dataset in this section
#also let's try login in to huggin face
#this indicates the model name and dataset name
from huggingface_hub import login
from google.colab import userdata
model_name="Qwen/Qwen2-1.5B-Instruct"
dataset_name="KisanVaani/agriculture-qa-english-only"
access_token=userdata.get("HF_TOKEN")
login(access_token)

In [ ]:
#now let's load our dataset and do some data exploration to understand our dataset
from datasets import load_dataset
dataset = load_dataset(dataset_name)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

In [ ]:
# i choose random 8000 samples instead of 23000 due to computational issue.i need faster training
dataset["train"]=dataset["train"].shuffle(seed=42).select(range(8000))
print(dataset)
print(dataset.column_names)
print(dataset['train'][0])
print(len(dataset["train"]))
#

DatasetDict({
    train: Dataset({
        features: ['question', 'answers'],
        num_rows: 8000
    })
})
{'train': ['question', 'answers']}
{'question': 'How can Tip over or Heart rot in banana plants be managed?', 'answers': 'Effective management of Tip over or Heart rot involves planting disease-free suckers, removing infected plants, and destroying them. Additionally, drenching with Methoxy ethyl mercuric chloride (Emisan-6) at 0.1%, Sodium hypochlorite at 10%, or Bleaching powder at 20g/litre/tree can help control the disease.'}
8000


In [ ]:
from models.qlora_model import QLoRAConfig

In [ ]:
from models.qlora_model import QLoRAModelLoader

In [ ]:
config = QLoRAConfig()
loader = QLoRAModelLoader(config)
model, tokenizer = loader.load()
model.print_trainable_parameters()


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|im_end|>
151645


In [ ]:
from utils.formatting import formatting_function as canonical_formatting_function

def notebook_formatting_function(example):
    return canonical_formatting_function(example, tokenizer)

In [ ]:
from datasets import DatasetDict

formatted_dataset = dataset["train"].map(
    lambda example: {"text": notebook_formatting_function(example)},
    remove_columns=dataset["train"].column_names,
)
formatted_dataset = DatasetDict({"train": formatted_dataset})

In [ ]:
# # formatted_dataset[0:15]
# print(type(formatted_dataset))
# print(formatted_dataset.column_names)

In [ ]:
# let's create folder to store our ouptut and checkpoints
from pathlib import Path
drive_output_dir: str = "/content/drive/MyDrive/agri_assistant/qwen_agriculture_lora"
local_checkpoint_dir: str = "/content/drive/MyDrive/agri_assistant/checkpoints"
Path(drive_output_dir).mkdir(parents=True, exist_ok=True)
Path(local_checkpoint_dir).mkdir(parents=True, exist_ok=True)

The model and tokenizer have been loaded successfully. Next, we can proceed with data preparation and fine-tuning.

In [ ]:
from models.trainer import TrainingConfig

In [ ]:
from models.trainer import QLoRATrainer

In [ ]:
# # this section checks for if tokenizer argument is supported in this our version
# from inspect import signature
# from trl import SFTTrainer

# print(signature(SFTTrainer.__init__))

In [ ]:
# from inspect import signature
# from trl import SFTConfig

# print(signature(SFTConfig.__init__))

In [ ]:
training_config = TrainingConfig()
trainer = QLoRATrainer(training_config, model, tokenizer, formatted_dataset)
trainer = trainer.build_trainer()
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026/08/07 14:43:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/07 14:43:38 INFO mlflow.store.db.utils: Updating database tables
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,3.024800,3.060881,1.766751,459693.000000,0.532484
2,2.954900,3.060881,1.766751,919386.000000,0.532484


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to

TrainOutput(global_step=900, training_loss=2.9188245709737144, metrics={'train_runtime': 8862.3427, 'train_samples_per_second': 1.625, 'train_steps_per_second': 0.102, 'total_flos': 9311949161717760.0, 'train_loss': 2.9188245709737144, 'epoch': 2.0})

In [ ]:
import os
adapter_dir = os.path.join(drive_output_dir, "adapter")
tokenizer_dir = os.path.join(drive_output_dir, "tokenizer")

trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(tokenizer_dir)
# training_args.to_json_file(os.path.join(cfg.drive_output_dir, "training_args.json"))
trainer.state.save_to_json(os.path.join(drive_output_dir, "trainer_state.json"))

expected_files = [
    os.path.join(adapter_dir, "adapter_model.safetensors"),
    os.path.join(adapter_dir, "adapter_config.json"),
    os.path.join(tokenizer_dir, "tokenizer_config.json"),
    # os.path.join(drive_output_dir, "training_args.json"),
    os.path.join(drive_output_dir, "trainer_state.json"),
]

print("Saved artifacts:")
for f in expected_files:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}] {f}")

Saved artifacts:
  [OK] /content/drive/MyDrive/agri_assistant/qwen_agriculture_lora/adapter/adapter_model.safetensors
  [OK] /content/drive/MyDrive/agri_assistant/qwen_agriculture_lora/adapter/adapter_config.json
  [OK] /content/drive/MyDrive/agri_assistant/qwen_agriculture_lora/tokenizer/tokenizer_config.json
  [OK] /content/drive/MyDrive/agri_assistant/qwen_agriculture_lora/trainer_state.json
